# Assessment Solutions - Do-Calculus and Valid Adjustment

In [ ]:
#!pip install pyro-ppl

In [ ]:
import torch
import pyro
import pyro.distributions as dist

Z_alias = {'unhappy', 'happy'}
X_alias = {'promotion0', 'promotion1'}
Y_alias = {'norenew', 'renew'}

N = 70000
N_happy_promo0 = 26231
N_happy_promo1 = 8128
N_unhappy_promo0 = 8769
N_unhappy_promo1 = 26872
N_happy = N_happy_promo0 + N_happy_promo1
N_unhappy = N_unhappy_promo0 + N_unhappy_promo1

# P(Z=unhappy), P(Z=happy)
prob_Z = torch.tensor([N_unhappy/N, N_happy/N])

# P(X=promo0 |Z= unhappy), P(X=promo1 |Z= unhappy)
# P(X=promo0 |Z= happy), P(X=promo1 |Z= happy)
prob_X = torch.tensor([
    [N_unhappy_promo0/N_unhappy, N_unhappy_promo1/N_unhappy],
    [N_happy_promo0/N_happy, N_happy_promo1/N_happy]
])

# P(Y=norenew | X=promo0, Z=unhappy), P(Y=renew | X=promo0, Z=unhappy)
# P(Y=norenew | X=promo0, Z=happy), P(Y=renew | X=promo0, Z=happy)

# P(Y=norenew | X=promo1, Z=unhappy), P(Y=renew | X=promo1, Z=unhappy)
# P(Y=norenew | X=promo1, Z=happy), P(Y=renew | X=promo1, Z=happy)
prob_Y = torch.tensor([
    [
        [0.068, 0.932],
        [0.267, 0.733]
    ],
    [
        [0.131, 0.869],
        [0.313, 0.687]
    ]
])  # P(Y|X,Z)

def promo_model():
    Z = pyro.sample('Z', dist.Categorical(probs=prob_Z))
    X = pyro.sample('X', dist.Categorical(probs=prob_X[Z]))
    Y = pyro.sample('Y', dist.Categorical(probs=prob_Y[X][Z]))
    return {'Z': Z, 'X': X, 'Y': Y}

Create the mutilated model for X = 1 (promotion).

In [ ]:
promo_1_model = pyro.do(promo_model, {'X': torch.tensor(1)})

Create the mutilated model for X = 0 (no promotion).

In [ ]:
promo_0_model = pyro.do(promo_model, {'X': torch.tensor(0)})

Now run a Monte Carlo simulator that simulates 100000 samples.

In [ ]:
def simulator():
    with pyro.plate("samples", 100000):
        y_1 = promo_1_model()['Y']
        y_0 = promo_0_model()['Y']
    return y_1 - y_0

torch.mean(simulator().double())

tensor(-0.0547, dtype=torch.float64)